In [ ]:
!git clone https://github.com/modelscope/DiffSynth-Studio.git


Cloning into 'DiffSynth-Studio'...
remote: Enumerating objects: 8894, done.
remote: Counting objects: 100% (2080/2080), done.
remote: Compressing objects: 100% (753/753), done.
remote: Total 8894 (delta 1397), reused 1327 (delta 1327), pack-reused 6814 (from 3)
Receiving objects: 100% (8894/8894), 15.40 MiB | 25.27 MiB/s, done.
Resolving deltas: 100% (5926/5926), done.


In [ ]:
import sys, os

repo_path = "/content/DiffSynth-Studio"
sys.path.insert(0, repo_path)

print(repo_path in sys.path)


True


In [ ]:
%%capture
%cd DiffSynth-Studio
!pip install -e .


In [ ]:
import diffsynth
print(diffsynth.__file__)


/content/DiffSynth-Studio/diffsynth/__init__.py


In [ ]:
# import diffsynth
from diffsynth.pipelines.qwen_image import (
    QwenImagePipeline, ModelConfig,
    QwenImageUnit_Image2LoRAEncode, QwenImageUnit_Image2LoRADecode
)
from modelscope import snapshot_download
from safetensors.torch import save_file
import torch
from PIL import Image

In [ ]:
vram_config_disk_offload = {
    "offload_dtype": "disk",
    "offload_device": "disk",
    "onload_dtype": "disk",
    "onload_device": "disk",
    "preparing_dtype": torch.bfloat16,
    "preparing_device": "cuda",
    "computation_dtype": torch.bfloat16,
    "computation_device": "cuda",
}

# Load models
pipe = QwenImagePipeline.from_pretrained(
    torch_dtype=torch.bfloat16,
    device="cuda",
    model_configs=[
        ModelConfig(model_id="DiffSynth-Studio/General-Image-Encoders", origin_file_pattern="SigLIP2-G384/model.safetensors", **vram_config_disk_offload),
        ModelConfig(model_id="DiffSynth-Studio/General-Image-Encoders", origin_file_pattern="DINOv3-7B/model.safetensors", **vram_config_disk_offload),
        ModelConfig(model_id="DiffSynth-Studio/Qwen-Image-i2L", origin_file_pattern="Qwen-Image-i2L-Style.safetensors", **vram_config_disk_offload),
    ],
    processor_config=ModelConfig(model_id="Qwen/Qwen-Image-Edit", origin_file_pattern="processor/"),
    vram_limit=torch.cuda.mem_get_info("cuda")[1] / (1024 ** 3) - 0.5,
)

2026-01-03 16:57:10,090 - modelscope - INFO - Got 1 files, start to download ...


Processing 1 items:   0%|          | 0.00/1.00 [00:00<?, ?it/s]

2026-01-03 16:59:34,604 - modelscope - INFO - Download model 'DiffSynth-Studio/General-Image-Encoders' successfully.


Loading models from: "./models/DiffSynth-Studio/General-Image-Encoders/SigLIP2-G384/model.safetensors"
Loaded model: {
    "model_name": "siglip2_image_encoder",
    "model_class": "diffsynth.models.siglip2_image_encoder.Siglip2ImageEncoder",
    "extra_kwargs": null
}


2026-01-03 16:59:36,992 - modelscope - INFO - Got 1 files, start to download ...


...

2026-01-03 17:06:42,678 - modelscope - INFO - Download model 'Qwen/Qwen-Image' successfully.


2026-01-03 17:06:45,399 - modelscope - INFO - Got 9 files, start to download ...


Processing 9 items:   0%|          | 0.00/9.00 [00:00<?, ?it/s]

2026-01-03 17:06:46,951 - modelscope - INFO - Download model 'Qwen/Qwen-Image-Edit' successfully.


Using siglip2_image_encoder from "./models/DiffSynth-Studio/General-Image-Encoders/SigLIP2-G384/model.safetensors".
Using dinov3_image_encoder from "./models/DiffSynth-Studio/General-Image-Encoders/DINOv3-7B/model.safetensors".
Using qwen_image_image2lora_style from "./models/DiffSynth-Studio/Qwen-Image-i2L/Qwen-Image-i2L-Style.safetensors".
No qwen_image_image2lora_coarse models available. This is not an error.
No qwen_image_image2lora_fine models available. This is not an error.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
images = []

import os

folder_path = "/content/drive/Shareddrives/NLP - Text2Img/ảnh trích sách lịch sử/luoc_su_nuoc_Viet_bang_tranh"
image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for filename in os.listdir(folder_path):
    if filename.lower().endswith(image_exts):
        img_path = os.path.join(folder_path, filename)
        images.append(Image.open(img_path))
len(images)



36

In [ ]:

# Model inference
with torch.no_grad():
    embs = QwenImageUnit_Image2LoRAEncode().process(pipe, image2lora_images=images)
    lora = QwenImageUnit_Image2LoRADecode().process(pipe, **embs)["lora"]
save_file(lora, "/content/drive/Shareddrives/NLP - Text2Img/Khuong/model_style.safetensors")

